In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv
import re
import nltk

In [2]:
data_train = pd.read_csv("../input/nlp-getting-started/train.csv")
data_test = pd.read_csv("../input/nlp-getting-started/test.csv")

In [3]:
data_train.head(15)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,8,NaN,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1
8,14,NaN,NaN,There's an emergency evacuation happening now ...,1
9,15,NaN,NaN,I'm afraid that the tornado is coming to our a...,1


In [4]:
data_test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [5]:
data_train.shape

(7613, 5)

In [6]:
data_train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [7]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [8]:
data_train.describe()

,id,target
count,7613.000000,7613.00000
mean,5441.934848,0.42966
std,3137.116090,0.49506
min,1.000000,0.00000
25%,2734.000000,0.00000
50%,5408.000000,0.00000
75%,8146.000000,1.00000
max,10873.000000,1.00000


In [9]:
data_train.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [10]:
data_train['keyword'].unique()

array([nan, 'ablaze', 'accident', 'aftershock', 'airplane%20accident',
       'ambulance', 'annihilated', 'annihilation', 'apocalypse',
       'armageddon', 'army', 'arson', 'arsonist', 'attack', 'attacked',
       'avalanche', 'battle', 'bioterror', 'bioterrorism', 'blaze',
       'blazing', 'bleeding', 'blew%20up', 'blight', 'blizzard', 'blood',
       'bloody', 'blown%20up', 'body%20bag', 'body%20bagging',
       'body%20bags', 'bomb', 'bombed', 'bombing', 'bridge%20collapse',
       'buildings%20burning', 'buildings%20on%20fire', 'burned',
       'burning', 'burning%20buildings', 'bush%20fires', 'casualties',
       'casualty', 'catastrophe', 'catastrophic', 'chemical%20emergency',
       'cliff%20fall', 'collapse', 'collapsed', 'collide', 'collided',
       'collision', 'crash', 'crashed', 'crush', 'crushed', 'curfew',
       'cyclone', 'damage', 'danger', 'dead', 'death', 'deaths', 'debris',
       'deluge', 'deluged', 'demolish', 'demolished', 'demolition',
       'derail', 'der

**Let's Train our Model Based on Training Data we had**

In [11]:
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
corpus = []
for i in range(0, 7613):
  text = re.sub('[^a-zA-Z]', ' ', data_train['text'][i])
  #text = text.lower()
  #text = text.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in text if not word in set(all_stopwords)]
  review = ' '.join(text)
  corpus.append(text)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 7613)
X = cv.fit_transform(corpus).toarray()
y = data_train.iloc[:, [4]].values

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

**first, let's try Naive Bayes classification**

In [14]:
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, np.ravel(y_train))

GaussianNB()

In [15]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 0]
 [1 0]
 [1 0]
 ...
 [1 1]
 [0 0]
 [1 1]]


In [16]:
#let's see the accuration
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[444 442]
 [120 517]]


0.6309914642153645

well, not bad, it gave 0.63 for accuration

**let's see how about applying the KNN algorithm**

In [17]:
from sklearn.neighbors import KNeighborsClassifier
classifier_k = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
classifier_k.fit(X_train, np.ravel(y_train))

KNeighborsClassifier()

In [18]:
y_prediction_knn= classifier_k.predict(X_test)
y_prediction_knn

array([0, 0, 0, ..., 0, 0, 0])

In [19]:
cm_knn = confusion_matrix(y_test, y_prediction_knn)
print(cm_knn)
accuracy_score(y_test, y_prediction_knn)

[[867  19]
 [454 183]]


0.6894287590282338

well, the KNN classification algorithm gave better accuracy, so let's use it to predict the test data

# **let's predict the test data using KNN **

In [20]:
corpus_train = []
for i in range(0, 3263): #I want to adjust the input dimension from the data train with the test data
  text = re.sub('[^a-zA-Z]', ' ', data_train['text'][i])
  text = text.lower()
  #text = text.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in text if not word in set(all_stopwords)]
  review = ' '.join(text)
  corpus_train.append(text)

In [21]:
#check the missing value and information of test data
data_test.isna().sum()

id             0
keyword       26
location    1105
text           0
dtype: int64

In [22]:
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3263 entries, 0 to 3262
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        3263 non-null   int64 
 1   keyword   3237 non-null   object
 2   location  2158 non-null   object
 3   text      3263 non-null   object
dtypes: int64(1), object(3)
memory usage: 102.1+ KB


In [23]:
data_test.describe()

,id
count,3263.000000
mean,5427.152927
std,3146.427221
min,0.000000
25%,2683.000000
50%,5500.000000
75%,8176.000000
max,10875.000000


In [24]:
data_test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [25]:
corpus_ = []
for i in range(0, 3263):
  text = re.sub('[^a-zA-Z]', ' ', data_test['text'][i])
  text = text.lower()
  #text = text.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in text if not word in set(all_stopwords)]
  review = ' '.join(text)
  corpus_.append(text)

In [26]:
cv_train = CountVectorizer(max_features = 3263)
X_train_new = cv_train.fit_transform(corpus_train).toarray()
y_train_new = data_train.iloc[:, [4]].values

In [27]:
cv_test = CountVectorizer(max_features = 3263)
X_ = cv_test.fit_transform(corpus_).toarray()

In [28]:
from sklearn.neighbors import KNeighborsClassifier
classifier_pr = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
classifier_pr.fit(X_train_new, np.ravel(y_train_new[0:3263]))

KNeighborsClassifier()

In [29]:
y_prediction=classifier_pr.predict(X_)
y_prediction

array([0, 0, 0, ..., 0, 0, 0])

In [30]:
submission=pd.DataFrame({'id': data_test['id'], 'target' : y_prediction.ravel()})
submission['target'] = submission['target']
submission.to_csv('submission.csv', index=False)
submission.tail()


,id,target
3258,10861,0
3259,10865,0
3260,10868,0
3261,10874,0
3262,10875,0


# **PS: actually I don't apply EDA so much to this code for this section :D, but I will aplly it next, so before we predict the tweet disaster classification using ML algorithm, at least we can gain insights from the available data first :) **